### Wave A — Lead volume & calendar

Mind-map sheets only (sample: Volume of leads per Day = Month × day-of-month).

| Sheet | Yellow box |
|-------|------------|
| `Volume_Per_Day` | #8 Monthly distribution (sample heatmap) |
| `Born_By_Source_Month` | #1 source attribution |
| `Born_By_FirstTouch_Channel` | #2 first-touch channel |
| `Born_Weekly` | #9 weekly lead count |
| `Bookings_By_DOW` | #7 bookings Mon–Sun |



In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path("/Users/lekhanigam/data/leads_sv_closure_reports")
sys.path.insert(0, str(ROOT))
from common import START_TS, END_TS, CLOSURE_STAGE, q, write_excel

print("Window pull…")
born = q(f"""
WITH ct AS (
  SELECT CAST(cid AS varchar) cid, MAX(CAST(source AS varchar)) src
  FROM campaigntrackers WHERE cid IS NOT NULL GROUP BY 1
)
SELECT
  CAST(l._id AS varchar) lead_oid,
  date_format(l.createdat, '%Y-%m') month_key,
  date_format(l.createdat, '%M') month_name,
  day(l.createdat) day_num,
  date_format(date_trunc('week', l.createdat), '%Y-%m-%d') week_start,
  TRIM(CAST(ls.name AS varchar)) source,
  COALESCE(NULLIF(TRIM(CAST(us.name AS varchar)), ''), TRIM(CAST(ls.name AS varchar))) channel
FROM leads l
LEFT JOIN leadsources ls ON CAST(ls._id AS varchar)=CAST(l.bornsource AS varchar)
LEFT JOIN ct ON ct.cid=TRIM(CAST(l.utm_campaign AS varchar))
LEFT JOIN utmsources us ON CAST(us._id AS varchar)=ct.src
WHERE l.createdat >= {START_TS} AND l.createdat < {END_TS}
""")

book = q(f"""
SELECT date_format(MIN(h.createdat), '%W') dow_name,
       day_of_week(MIN(h.createdat)) dow_num,
       CAST(h.lead AS varchar) lead_oid
FROM leadstagehistories h
WHERE CAST(h.stage AS varchar)='{CLOSURE_STAGE}'
  AND h.createdat >= {START_TS} AND h.createdat < {END_TS}
GROUP BY CAST(h.lead AS varchar)
""")

vol = born.groupby(["month_key", "month_name", "day_num"], as_index=False).agg(n=("lead_oid", "nunique"))
volume_per_day = (
    vol.pivot_table(index=["month_key", "month_name"], columns="day_num", values="n", aggfunc="sum", fill_value=0)
    .reset_index().sort_values("month_key")
)
volume_per_day = volume_per_day.rename(columns={"month_name": "Month"}).drop(columns=["month_key"])
for d in range(1, 32):
    if d not in volume_per_day.columns:
        volume_per_day[d] = 0
volume_per_day = volume_per_day[["Month"] + list(range(1, 32))]

sheets = {
    "Volume_Per_Day": volume_per_day,
    "Born_By_Source_Month": (
        born.groupby(["source", "month_name", "month_key"], as_index=False)
        .agg(**{"Lead count": ("lead_oid", "nunique")})
        .sort_values(["month_key", "Lead count"], ascending=[True, False])
        .drop(columns=["month_key"])
    ),
    "Born_By_FirstTouch_Channel": (
        born.groupby(["channel", "month_name", "month_key"], as_index=False)
        .agg(**{"Lead count": ("lead_oid", "nunique")})
        .sort_values(["month_key", "Lead count"], ascending=[True, False])
        .drop(columns=["month_key"])
        .rename(columns={"channel": "First-touch channel"})
    ),
    "Born_Weekly": (
        born.groupby(["week_start"], as_index=False)
        .agg(**{"Lead count": ("lead_oid", "nunique")})
        .sort_values("week_start")
    ),
    "Bookings_By_DOW": (
        book.groupby(["dow_num", "dow_name"], as_index=False)
        .agg(**{"Sales closure count": ("lead_oid", "nunique")})
        .sort_values("dow_num")
        .drop(columns=["dow_num"])
        .rename(columns={"dow_name": "Day of week"})
    ),
}
write_excel(ROOT / "Lead_Volume_Calendar_Analysis.xlsx", sheets)
print("born", len(born), "closures", book["lead_oid"].nunique())

